# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asadnaeem23/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
!git clone https://github.com/Asadnaeem23/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 134 (delta 45), reused 89 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 1.87 MiB | 9.81 MiB/s, done.
Resolving deltas: 100% (45/45), done.


In [4]:
%cd /content/flyrank-ml-internship

/content/flyrank-ml-internship


In [5]:
import os

print("Current folder:", os.getcwd())
print("Data exists:", os.path.exists("data/raw/content_refresh_anonymized.csv"))

Current folder: /content/flyrank-ml-internship
Data exists: True


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule and signal verdicts

I am using the Content Refresh lane.

The baseline will use two observed signals:

1. `days_since_last_update` — linked to content staleness and FlyRank's refresh-related flags.
2. `ctr` — observed search click-through performance.

#### Signal checks

- `days_since_last_update` — **MIXED**. The oldest bucket (`181+`) has very low median impressions and CTR, but the `91–180` bucket performs differently from the expected simple staleness pattern. Therefore, staleness is useful as a directional review signal but is not a clean standalone predictor.
- `ctr` — **MIXED**. Higher CTR buckets generally have better observed average position, but impressions do not change consistently with CTR. Therefore, CTR is useful as a directional search-performance signal but should not be treated as a standalone predictor.

#### Baseline rule

For each content item, assign points for:

- being older since its last update;
- having low CTR.

The total points form the baseline score. Higher scores receive higher refresh priority.

#### Reason code

The rule outputs one reason code:

- `REFRESH_STALENESS_CTR` — the item received priority because its observed staleness and/or low CTR contributed to the score.

#### Action

- `REVIEW_REFRESH` — high-scoring content should be reviewed for a possible refresh.
- `NO_ACTION` — lower-scoring content is not prioritized by this baseline.

This is a directional decision-support rule. It does not claim that refreshing the content will cause a performance improvement.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

# Load the Content Refresh starter data.
data_path = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)

# ---------------------------------------------------------
# Signal 1: days_since_last_update
# ---------------------------------------------------------

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30", "31-90", "91-180", "181+"]
)

staleness_check = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median"),
          median_ctr=("ctr", "median")
      )
      .reset_index()
)

print("\nSignal 1 — days_since_last_update")
display(staleness_check)


# ---------------------------------------------------------
# Signal 2: ctr
# ---------------------------------------------------------

df["ctr_bucket"] = pd.cut(
    df["ctr"],
    bins=[-float("inf"), 0.5, 1.0, 2.0, float("inf")],
    labels=["<0.5%", "0.5-1%", "1-2%", "2%+"]
)

ctr_check = (
    df.groupby("ctr_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median"),
          median_avg_position=("avg_position", "median")
      )
      .reset_index()
)

print("\nSignal 2 — ctr")
display(ctr_check)


Dataset shape: (30000, 44)

Signal 1 — days_since_last_update


,staleness_bucket,n,median_impressions,median_ctr
0,0-30,20480,470.0,0.04
1,31-90,175,510.0,0.00
2,91-180,9171,1692.0,0.10
3,181+,174,15.5,0.00



Signal 2 — ctr


,ctr_bucket,n,median_impressions,median_avg_position
0,<0.5%,25851,712.0,11.5
1,0.5-1%,2460,2677.5,8.5
2,1-2%,915,712.0,8.4
3,2%+,774,22.0,6.1


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Baseline scoring rule

The baseline score uses only information available at the decision moment.

Scoring:

- Add **2 points** when `days_since_last_update` is greater than 180 days.
- Add **1 point** when `days_since_last_update` is between 91 and 180 days.
- Add **1 point** when `ctr` is below 0.5%.
- Otherwise, add 0 points for that signal.

The final score is the sum of these points.

Action labels:

- `REVIEW_REFRESH` — score >= 2.
- `NO_ACTION` — score < 2.

Reason code:

- `REFRESH_STALENESS_CTR` — the item received priority because staleness and/or low CTR contributed to its score.

The score is a simple directional baseline for prioritizing content review. It is not a prediction of causal refresh impact.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2: Build the ranked baseline queue.

import os
import pandas as pd

# Use the same Content Refresh dataset used in Section 1.
data_path = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

# Calculate points from staleness.
df["staleness_points"] = 0

df.loc[
    df["days_since_last_update"].between(91, 180, inclusive="both"),
    "staleness_points"
] = 1

df.loc[
    df["days_since_last_update"] > 180,
    "staleness_points"
] = 2

# Calculate points from low CTR.
df["ctr_points"] = (
    df["ctr"] < 0.5
).astype(int)

# Final baseline score.
df["baseline_score"] = (
    df["staleness_points"] +
    df["ctr_points"]
)

# Action label.
df["action"] = df["baseline_score"].apply(
    lambda x: "REVIEW_REFRESH" if x >= 2 else "NO_ACTION"
)

# One reason code.
df["reason_code"] = df["baseline_score"].apply(
    lambda x: "REFRESH_STALENESS_CTR"
    if x >= 2
    else "NO_PRIORITY_SIGNAL"
)

# Rank all content items.
df = df.sort_values(
    ["baseline_score", "days_since_last_update", "ctr"],
    ascending=[False, False, True]
).reset_index(drop=True)

df["rank"] = df.index + 1

# Keep useful fields in the output queue.
queue_columns = [
    "rank",
    "content_id",
    "client_id",
    "baseline_score",
    "action",
    "reason_code",
    "days_since_last_update",
    "ctr",
    "impressions_90d"
]

queue = df[queue_columns].copy()

# Write the required CSV.
output_path = "work/outputs/baseline_action_score.csv"
os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(output_path, index=False)

print(f"Queue written to: {output_path}")
print(f"Rows written: {len(queue):,}")

print("\nTop 20 rows:")
display(queue.head(20))


Queue written to: work/outputs/baseline_action_score.csv
Rows written: 30,000

Top 20 rows:


,rank,content_id,client_id,baseline_score,action,reason_code,days_since_last_update,ctr,impressions_90d
0,1,content_55a5b1c46474,client_4ec9599fc2,3,REVIEW_REFRESH,REFRESH_STALENESS_CTR,373,0.0,35
1,2,content_f6fdf87348f6,client_4ec9599fc2,3,REVIEW_REFRESH,REFRESH_STALENESS_CTR,373,0.0,2
2,3,content_8d56efff1e71,client_4ec9599fc2,3,REVIEW_REFRESH,REFRESH_STALENESS_CTR,372,0.0,1
3,4,content_1b4ec72dafd4,client_4ec9599fc2,3,REVIEW_REFRESH,REFRESH_STALENESS_CTR,372,0.0,2
4,5,content_e2b702f4f92b,client_4ec9599fc2,3,REVIEW_REFRESH,REFRESH_STALENESS_CTR,334,0.0,30
5,6,content_06e19c6486b0,client_4ec9599fc2,3,REVIEW_REFRESH,REFRESH_STALENESS_CTR,334,0.0,10
6,7,content_7a888d3d99c8,client_19581e27de,3,REVIEW_REFRESH,REFRESH_STALENESS_CTR,313,0.0,95
7,8,content_6476d1d8c050,client_19581e27de,3,REVIEW_REFRESH,REFRESH_STALENESS_CTR,313,0.0,304
8,9,content_94991fe6268c,client_19581e27de,3,REVIEW_REFRESH,REFRESH_STALENESS_CTR,313,0.0,7
9,10,content_02b0d6e30129,client_19581e27de,3,REVIEW_REFRESH,REFRESH_STALENESS_CTR,313,0.0,176


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

The top 20 items are reviewed as decision-support recommendations, not confirmed refresh requirements.

Each item receives the same action and reason code generated by the baseline rule. The review also records what could make the recommendation wrong, such as unavailable data, unusual content circumstances, or the simple rule not capturing the item's actual business context.

The baseline is intentionally conservative: these are items to review, not items that must be refreshed.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: Review the top 20 ranked items.

top20 = queue.head(20).copy()

top20["why_selected"] = top20.apply(
    lambda row: (
        f"Score {row['baseline_score']} from "
        f"{row['days_since_last_update']} days since last update "
        f"and CTR of {row['ctr']:.2f}%."
    ),
    axis=1
)

top20["what_would_make_it_wrong"] = (
    "The measured signals may not reflect the current content context; "
    "manual review could show that a refresh is not appropriate."
)

review_columns = [
    "rank",
    "content_id",
    "client_id",
    "action",
    "reason_code",
    "why_selected",
    "what_would_make_it_wrong"
]

top20_review = top20[review_columns]

display(top20_review)

,rank,content_id,client_id,action,reason_code,why_selected,what_would_make_it_wrong
0,1,content_55a5b1c46474,client_4ec9599fc2,REVIEW_REFRESH,REFRESH_STALENESS_CTR,Score 3 from 373 days since last update and CT...,The measured signals may not reflect the curre...
1,2,content_f6fdf87348f6,client_4ec9599fc2,REVIEW_REFRESH,REFRESH_STALENESS_CTR,Score 3 from 373 days since last update and CT...,The measured signals may not reflect the curre...
2,3,content_8d56efff1e71,client_4ec9599fc2,REVIEW_REFRESH,REFRESH_STALENESS_CTR,Score 3 from 372 days since last update and CT...,The measured signals may not reflect the curre...
3,4,content_1b4ec72dafd4,client_4ec9599fc2,REVIEW_REFRESH,REFRESH_STALENESS_CTR,Score 3 from 372 days since last update and CT...,The measured signals may not reflect the curre...
4,5,content_e2b702f4f92b,client_4ec9599fc2,REVIEW_REFRESH,REFRESH_STALENESS_CTR,Score 3 from 334 days since last update and CT...,The measured signals may not reflect the curre...
5,6,content_06e19c6486b0,client_4ec9599fc2,REVIEW_REFRESH,REFRESH_STALENESS_CTR,Score 3 from 334 days since last update and CT...,The measured signals may not reflect the curre...
6,7,content_7a888d3d99c8,client_19581e27de,REVIEW_REFRESH,REFRESH_STALENESS_CTR,Score 3 from 313 days since last update and CT...,The measured signals may not reflect the curre...
7,8,content_6476d1d8c050,client_19581e27de,REVIEW_REFRESH,REFRESH_STALENESS_CTR,Score 3 from 313 days since last update and CT...,The measured signals may not reflect the curre...
8,9,content_94991fe6268c,client_19581e27de,REVIEW_REFRESH,REFRESH_STALENESS_CTR,Score 3 from 313 days since last update and CT...,The measured signals may not reflect the curre...
9,10,content_02b0d6e30129,client_19581e27de,REVIEW_REFRESH,REFRESH_STALENESS_CTR,Score 3 from 313 days since last update and CT...,The measured signals may not reflect the curre...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks and leakage check

Some top-ranked items may be weak recommendations when their CTR is based on very few impressions. In those cases, a zero or very low CTR may be unstable and should be manually reviewed before acting.

The baseline score uses only `days_since_last_update` and `ctr`.

I did not use trend fields, product/model fields, or future-window information in the scoring rule. The result is a decision-support ranking, not a causal prediction of refresh impact.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Weak-pick audit and leakage check.

# Weak picks: very low impression volume makes CTR less reliable.
weak_picks = queue[
    (queue["rank"] <= 20) &
    (queue["impressions_90d"] < 10)
].copy()

print("Weak top-20 picks with fewer than 10 impressions:")
display(
    weak_picks[
        [
            "rank",
            "content_id",
            "baseline_score",
            "days_since_last_update",
            "ctr",
            "impressions_90d",
            "action",
            "reason_code"
        ]
    ]
)

# Confirm exactly which fields the baseline score uses.
score_inputs = {
    "days_since_last_update",
    "ctr"
}

# Fields that should not be used as future/label-derived signals.
forbidden_inputs = {
    "trend_direction",
    "trend_pct",
    "provider_used",
    "model_used"
}

leaked_inputs = score_inputs.intersection(forbidden_inputs)

print("\nScoring inputs:", sorted(score_inputs))
print("Forbidden inputs used:", sorted(leaked_inputs))

assert leaked_inputs == set()

print("\nLeakage check: PASS")
print("The baseline score uses only the two intended observed signals.")

Weak top-20 picks with fewer than 10 impressions:


,rank,content_id,baseline_score,days_since_last_update,ctr,impressions_90d,action,reason_code
1,2,content_f6fdf87348f6,3,373,0.0,2,REVIEW_REFRESH,REFRESH_STALENESS_CTR
2,3,content_8d56efff1e71,3,372,0.0,1,REVIEW_REFRESH,REFRESH_STALENESS_CTR
3,4,content_1b4ec72dafd4,3,372,0.0,2,REVIEW_REFRESH,REFRESH_STALENESS_CTR
8,9,content_94991fe6268c,3,313,0.0,7,REVIEW_REFRESH,REFRESH_STALENESS_CTR
12,13,content_ccf25ed65a99,3,305,0.0,2,REVIEW_REFRESH,REFRESH_STALENESS_CTR
14,15,content_129753e3095f,3,305,0.0,7,REVIEW_REFRESH,REFRESH_STALENESS_CTR
17,18,content_ab18b5811c02,3,305,0.0,5,REVIEW_REFRESH,REFRESH_STALENESS_CTR



Scoring inputs: ['ctr', 'days_since_last_update']
Forbidden inputs used: []

Leakage check: PASS
The baseline score uses only the two intended observed signals.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.